<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

In [135]:
# Chapter 2: Working with Text Data

Packages that are being used in this notebook:

In [136]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path = "the-verdict.txt"
    urrllib.request.urlretrieve(url, file_path)


In [137]:
with open("the-verdict.txt", "r", encoding="UTF-8") as f:
    raw_text = f.read()

In [138]:
#raw_text

In [139]:
len(raw_text)

20479

In [140]:
import re

text = "Hello, world. This, is a tests."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'tests.']


In [141]:
result = re.split(r'([,.]\s)', text)
print(result)

['Hello', ', ', 'world', '. ', 'This', ', ', 'is a tests.']


In [142]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ', ', 'world', '. ', 'This', ', ', 'is a tests.']


In [143]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result
#print(preprocessed)

In [144]:
len(preprocessed)

4690

## 2.3 Converting tokens into token IDs

In [169]:
vocab["Jack"] # Token ID for "Jack"

57

In [170]:
int_to_str = {i:s for s, i in vocab.items()}

int_to_str[57]

'Jack'

In [171]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [196]:
vocab = {token:integer for integer, token in enumerate(all_words)}
#vocab

In [197]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Make Token IDs every token in 'preprocessed'
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """
        Decode the tokenized text back to 'preprocessed'
        """
        
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [198]:
tokenizer = SimpleTokenizerV1(vocab)

In [199]:
text = """ "It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""

In [200]:
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [201]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

In [202]:
text = "Hello, do you like tea? is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

In [203]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [204]:
len(vocab.items())

1132

In [205]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [210]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        
        # Make Token IDs every token in 'preprocessed'
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """
        Decode the tokenized text back to 'preprocessed'
        """
        
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [211]:
tokenizer = SimpleTokenizerV2(vocab)

In [212]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 584, 999, 6, 115, 1131, 10]